### Simulator

In [ ]:
import numpy as np
import random
import math
import csv


class TPM:
    """
    Tree Parity Machine with Hebbian learning.
    """

    def __init__(self, K, N, L, weights=None, scale=1):
        self.K = K
        self.N = N
        self.L = L
        self.scale = scale
        if weights is None:
            self.w = np.random.randint(-L, L + 1, size=(K, N), dtype=int) * scale
        else:
            self.w = np.array(weights, dtype=int).reshape(K, N)

    @staticmethod
    def _sign(x):
        return np.where(x >= 0, 1, -1)

    def forward(self, x):
        h = np.sum(self.w * x, axis=1)
        sigma = self._sign(h)
        tau = np.prod(sigma)
        return tau, sigma

    def hebbian_update(self, x, tau_self, sigma_self, tau_partner):
        if tau_self != tau_partner:
            return

        mask_hidden = (sigma_self == tau_self)
        if not np.any(mask_hidden):
            return

        mask = mask_hidden[:, np.newaxis]
        delta = tau_self * x * mask
        self.w = self.w + delta
        np.clip(self.w, -self.L * self.scale, self.L * self.scale, out=self.w)


def bits_to_weights(bitstring, K, N, L, scale=1):
    num_weights = K * N
    alphabet_size = 2 * L + 1
    bits_per_weight = math.ceil(math.log2(alphabet_size))

    expected_len = num_weights * bits_per_weight
    if len(bitstring) < expected_len:
        raise ValueError("Bitstring too short")

    weights = []
    idx = 0
    for _ in range(num_weights):
        val = 0
        for _ in range(bits_per_weight):
            val = (val << 1) | bitstring[idx]
            idx += 1
        mapped = (val % alphabet_size) - L
        weights.append(mapped * scale)

    return np.array(weights, dtype=int).reshape(K, N)


def random_bitstring(length):
    return [random.randint(0, 1) for _ in range(length)]


def make_correlated_tpm_triplet(K, N, L, qber, scale=1):
    alphabet_size = 2 * L + 1
    bits_per_weight = math.ceil(math.log2(alphabet_size))
    total_bits = K * N * bits_per_weight

    bits_A = random_bitstring(total_bits)

    bits_B = bits_A.copy()
    num_flip = int(qber * total_bits)
    flip_positions = random.sample(range(total_bits), num_flip) if num_flip > 0 else []
    for pos in flip_positions:
        bits_B[pos] ^= 1

    bits_E = random_bitstring(total_bits)

    A = TPM(K, N, L, weights=bits_to_weights(bits_A, K, N, L, scale=scale), scale=scale)
    B = TPM(K, N, L, weights=bits_to_weights(bits_B, K, N, L, scale=scale), scale=scale)
    E = TPM(K, N, L, weights=bits_to_weights(bits_E, K, N, L, scale=scale), scale=scale)

    return A, B, E


def enforce_no_zero(arr, M, rng):
    """
    Replace zeros by resampled values from nonzero integer range.
    """
    out = arr.copy()
    zero_mask = (out == 0)
    if np.any(zero_mask):
        candidates = np.concatenate((np.arange(-M, 0), np.arange(1, M + 1)))
        out[zero_mask] = rng.choice(candidates, size=np.sum(zero_mask))
    return out


# ============================================================
# Main generate_input with new method
# ============================================================

def generate_input(
    method,
    K,
    N,
    M,
    rng,

):

    # Standard zero-free uniform
    if method == "uniform":
        candidates = np.concatenate((np.arange(-M, 0), np.arange(1, M + 1)))
        x = rng.choice(candidates, size=(K, N))
        return x


    

# ============================================================
# Sync logic
# ============================================================

def run_single_sync(
    K, N, L, M, qber,
    input_method="uniform",
    max_steps=10_000_000,
    rng=None,
    scale=1
):

    if rng is None:
        rng = np.random.default_rng()

    A, B, E = make_correlated_tpm_triplet(K, N, L, qber, scale=scale)

    A_init = A.w.copy().flatten().tolist()
    B_init = B.w.copy().flatten().tolist()
    E_init = E.w.copy().flatten().tolist()

    steps = 0

    while not np.array_equal(A.w, B.w):
        steps += 1
        if steps > max_steps:
            break

        x = generate_input(
            method=input_method,
            K=K,
            N=N,
            M=M,
            rng=rng,
        )

        tau_A, sigma_A = A.forward(x)
        tau_B, sigma_B = B.forward(x)
        tau_E, sigma_E = E.forward(x)


        if tau_A == tau_B:
            A.hebbian_update(x, tau_A, sigma_A, tau_B)
            B.hebbian_update(x, tau_B, sigma_B, tau_A)
            if tau_E == tau_A:
                E.hebbian_update(x, tau_E, sigma_E, tau_A)

    A_B_synced = np.array_equal(A.w, B.w)

    if not A_B_synced:
        E_success = False
        E_match_percent = 0.0
    else:
        matches = np.sum(E.w == A.w)
        total = K * N
        E_match_percent = 100.0 * matches / total
        E_success = (matches == total)

    return (
        steps,
        E_success,
        E_match_percent,
        A_init,
        B_init,
        E_init,
        A.w.copy().flatten().tolist(),
        E.w.copy().flatten().tolist()
    )


# ============================================================
# Batch runner
# ============================================================

def run_experiments(
    K, N, L, M,
    qber,
    n_runs,
    csv_path,
    seed=None,
    input_method="uniform",
    scale=1
):

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    rng = np.random.default_rng(seed)

    fieldnames = [
        "N", "K", "L", "M",
        "qber",
        "input_method",
        "steps",
        "E_success",
        "E_match_percent",
        "A_init",
        "B_init",
        "E_init",
        "A_final",
        "E_final"
    ]

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for _ in range(n_runs):
            (
                steps,
                E_success,
                E_match_percent,
                A_init,
                B_init,
                E_init,
                A_final,
                E_final
            ) = run_single_sync(
                K=K,
                N=N,
                L=L,
                M=M,
                qber=qber,
                input_method=input_method,
                rng=rng,
                scale=scale
            )
            print(f"{_}/{n_runs}")
            writer.writerow({
                "N": N, "K": K, "L": L, "M": M,
                "qber": qber,
                "input_method": input_method,
                "steps": steps,
                "E_success": bool(E_success),
                "E_match_percent": E_match_percent,
                "A_init": A_init,
                "B_init": B_init,
                "E_init": E_init,
                "A_final": A_final,
                "E_final": E_final
            })



### Usage (matching experiment in the paper setup)

In [ ]:
l=5
for c in [0,1,2,3,4,5,6,7,8,9,10,11]:
    scale=2**c
    for m in [1,2,3,4,5]:
        print(f"Running experiments for M={m}, L={l}")
        for q in [0.05]:
            print(f"  QBER={q}")
            run_experiments(K=3, N=60, L=l, M=m*scale, qber=q, n_runs=10000, csv_path=f"results/results_q0.05_m{m*scale}_l{l*scale}_uniform.csv", scale=scale,seed=42, input_method="uniform")


#### c, n_runs, m, N, K can be changed to run different sets of experiments. Note that scale is 2**c.